# Day 2: answering the question

**Python for Data Analysis**, Coding for Reproducible Research (CfRR), University of Exeter

Yesterday we cleaned every month of Oxford weather since 1853 and found that summer 2026 tops the ranking. Today we find out whether that means anything: we plot it, fit a trend and test how sure we can be of it, bring in a second station and combine the two records, try a first machine-learning model, and make sure the whole analysis can be rerun by someone else.

## By the end of today you will be able to

- build a matplotlib figure step by step with `fig, ax`, and save it as an image
- make interactive charts with Plotly, and say when each library is the better tool
- say what a NumPy array is, why vectorised code beats a loop, when to reach for a NumPy function rather than a pandas method, and how `NaN` behaves in each
- fit a straight-line trend, say how sure you can be of it, and what it cannot tell you
- turn repeated steps into a function, and move it into a `.py` file that any notebook can import
- combine tables with `concat` and `merge`, save them to CSV, and reshape them from long to wide
- fit and test a first scikit-learn model, check it against a baseline, and say why it must be tested on data it has not seen

## How to use this notebook

As yesterday, there are two versions. In `day2.ipynb` most code cells are empty, and you type the code into them as the instructor types it on screen; the few cells that are filled in already, just run. [`day2_solutions.ipynb`](day2_solutions.ipynb), in the same folder, has every cell filled in, with its output: copy any cell you miss from there.

*Data: Met Office historic station data for Oxford and Lerwick. Contains public sector information licensed under the [Open Government Licence v3.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).*

## 1. Recap

*10:00, 10 minutes*

The first two cells are filled in for you. The first imports today's libraries and names the data file and its columns. The second is the whole of yesterday's cleaning in a dozen lines, so everyone starts with the same DataFrame. It is called `oxford` rather than `df`, because a second station joins it later. Run them both, then read the second one through: every line is something we did on Day 1 except one, the new `tmean` column, which block 3 comes back to.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

DATA = Path("data")
# Snapshot of https://www.metoffice.gov.uk/pub/data/weather/uk/climate/stationdata/oxforddata.txt
OXFORD = DATA / "oxforddata.txt"
COLUMNS = ["yyyy", "mm", "tmax", "tmin", "af", "rain", "sun", "status"]
MEASURES = ["tmax", "tmin", "af", "rain", "sun"]

In [ ]:
oxford = pd.read_csv(OXFORD, sep=r"\s+", skiprows=7, names=COLUMNS, na_values="---")
for col in MEASURES:
    text = oxford[col].astype(str)
    oxford[col + "_estimated"] = text.str.endswith("*")
    oxford[col] = text.str.rstrip("*").astype(float)
oxford["provisional"] = oxford["status"] == "Provisional"
oxford = oxford.drop(columns="status")
oxford["tmean"] = (oxford["tmax"] + oxford["tmin"]) / 2
oxford["date"] = pd.to_datetime({"year": oxford["yyyy"], "month": oxford["mm"], "day": 1})
oxford = oxford.set_index("date")
oxford.shape

Then yesterday's summer averages again, with one change: `reset_index()` turns the years from the index back into an ordinary column, `yyyy`. From here on, every number we use has a column name.

## 2. Seeing it

*10:10, 40 minutes*

A matplotlib chart has two main objects: the **Figure** (the whole image) and the **Axes** (one plot inside it, with its own axes, labels and lines). Every chart here starts with `fig, ax = plt.subplots()` and then adds things to `ax`. Writing it this way takes a few more lines than a one-liner, but you control everything. The [CfRR Matplotlib page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_Matplotlib.html) has many more chart types.

Start with every month in the record.

Unreadable: the seasonal cycle swamps everything else. Choose the chart for the question. Ours is about summers, so plot one value per summer.

(Why not the mean for each calendar year? 2026 only has nine months so far. The missing three, October to December, are cooler than the year's average, so leaving them out would flatter it.)

Year-to-year noise hides any slow change. A 10-year rolling mean averages each summer with the nine before it. Store it as a new column, `rolling`, then add it as a second, thicker line, with a legend to tell them apart.

Because we kept hold of `ax`, we can keep adding to the same chart. Highlight 2026, then save the figure as a file you could put in a paper. `.loc` with a mask picks out 2026's `tmax`, as it did yesterday, and `.item()` takes the single number out of it.

### The world's your oyster with matplotlib

Everything so far has been one chart on one `ax`. A figure can hold as many as you like, laid out however you like, and every part of each one can be changed. The cell below is filled in: run it, then read it for the ideas rather than every line.

- `plt.subplot_mosaic` lays out the figure from a sketch of it. Each inner list is a row, and a name used more than once makes one `ax` span those places, so `"heat"` runs the full height and `"summers"` two columns. `axes` is a dictionary: `axes["heat"]` is an `ax` like the ones above. `layout="constrained"` stops titles and labels overlapping.
- The heatmap: `unstack` moves the months into the columns, one row per year (section 5 comes back to it). `grid.index // 10 * 10` turns each year into its decade: `//` divides and drops the remainder, so 1987 `// 10` is 198, and times 10 is 1980. `pcolormesh` colours each cell by its value, and `fig.colorbar` adds the key.
- `annotate` adds text with an arrow, `fill_between` shades between two lines, `bar` takes a colour for each bar, and the last loop takes the top and right edges off every `ax`.

The [matplotlib gallery](https://matplotlib.org/stable/gallery/index.html) has hundreds more examples, each with its code.

In [ ]:
months = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
fig, axes = plt.subplot_mosaic([["heat", "summers", "summers"],
                                ["heat", "cycle", "2026"]],
                               figsize=(14, 7), layout="constrained")
fig.suptitle("Oxford since 1853", fontsize=18, fontweight="bold")

# Every month of the record, averaged by decade
grid = oxford.set_index(["yyyy", "mm"])["tmax"].unstack("mm")
decades = grid.groupby(grid.index // 10 * 10).mean()
mesh = axes["heat"].pcolormesh(decades.columns, decades.index, decades, cmap="Reds")
fig.colorbar(mesh, ax=axes["heat"], label="Average daily max (°C)")
axes["heat"].set(title="Every month, by decade", ylabel="Decade")

# Every summer, the rolling mean, and 2026 picked out
axes["summers"].plot(summer_tmax["yyyy"], summer_tmax["tmax"], color="grey", linewidth=0.8, label="Each summer")
axes["summers"].plot(summer_tmax["yyyy"], summer_tmax["rolling"], color="black", linewidth=2.5, label="10-year rolling mean")
axes["summers"].scatter(2026, tmax_2026, color="red", zorder=3)
axes["summers"].annotate(f"2026: {tmax_2026:.1f} °C", xy=(2026, tmax_2026), xytext=(1970, 25.8),
                         color="red", fontweight="bold", arrowprops={"arrowstyle": "->", "color": "red"})
axes["summers"].set(title="Every summer", ylabel="June to August max (°C)")
axes["summers"].legend(loc="upper left")

# The year's cycle, then and now, with the gap shaded
early = oxford[oxford["yyyy"] < 1900].groupby("mm")["tmax"].mean()
normal = oxford[oxford["yyyy"].between(1991, 2020)].groupby("mm")["tmax"].mean()
axes["cycle"].plot(early, color="tab:blue", label="1853 to 1899")
axes["cycle"].plot(normal, color="tab:red", label="1991 to 2020")
axes["cycle"].fill_between(normal.index, early, normal, color="tab:red", alpha=0.2)
axes["cycle"].set(title="The year, then and now", ylabel="Average daily max (°C)")
axes["cycle"].legend()

# 2026 so far, month by month, against 1991 to 2020
diff = (oxford[oxford["yyyy"] == 2026].set_index("mm")["tmax"] - normal).dropna()
axes["2026"].bar(diff.index, diff, color=["tab:red" if d > 0 else "tab:blue" for d in diff])
axes["2026"].axhline(0, color="black", linewidth=0.8)
axes["2026"].set(title="2026 so far, against 1991 to 2020", ylabel="Difference (°C)")

for name in ["heat", "cycle", "2026"]:
    axes[name].set_xticks(range(1, 13), labels=months)
for ax in axes.values():
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

### The same thing with Plotly

*Last 15 minutes of this block*

Plotly makes interactive charts that you can hover over, zoom and pan. `plotly.express` builds a whole chart in one call. The [CfRR Plotly page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/data-analysis-plotly.html) goes further.

`px.line` takes a table and the names of its columns, so `summer_tmax` goes straight in.

Hover over the points to find the three hottest summers. Drag across the 1970s to zoom in, and double-click to zoom back out.

Plotly has a function for each kind of chart, all used the same way: a table, then the names of columns. A **box plot** shows how values are spread. The box runs from the 25% value to the 75% value, as in yesterday's `.describe()`, the line across it is the median, and dots beyond the whiskers are unusual months. One box for each month of the year:

Hover over the dot above the July box, then the dots above August: `hover_data` adds the year.

Which tool, when?

- **Plotly:** quick to write and good for exploring; charts can be shared as web pages.
- **matplotlib:** more lines of code, but control over every detail, and it saves the static image files that journals want. (Plotly can save static images too, but needs the extra `kaleido` package and a copy of Chrome.)

## Break

*10:50, 5 minutes*

## 3. Is it a trend?

*10:55, 30 minutes*

To fit a trend we use NumPy, the library underneath pandas. First, a look at what it is.

### What's underneath: NumPy

*First 10 minutes of this block*

Every pandas column is a NumPy array with a name and an index attached. NumPy is what makes the arithmetic fast. The [CfRR NumPy page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_NumPy.html) has the full tour.

The cleaning cell at the top of this notebook gave `oxford` one column we didn't make yesterday, `tmean`: the mean temperature for each month, the average of its mean maximum and mean minimum. Here is that calculation the way the Introduction to Python course would write it, with a loop:

NumPy can do the same calculation without a loop. Arithmetic on an array applies to every element at once: this is called **vectorisation**.

Same numbers, less code, and much faster: on a typical laptop the vectorised version is over 100 times faster than the loop, and the gap grows with the size of the data. That is why pandas and NumPy code avoids looping over rows.

pandas columns work the same way. The cleaning cell made `tmean` with `(oxford["tmax"] + oxford["tmin"]) / 2`, which lines the values up by row, and yesterday's masks, such as `df["tmax"] > 20`, compared every month at once.

### NumPy's functions

NumPy also has constants, and maths functions that work on a whole array at once. Two constants first: `np.nan`, "not a number", the value pandas shows as `NaN` for a missing month, and `np.pi`.

Many NumPy functions have a pandas method that does the same job: `np.mean(x)` and `x.mean()`, `np.max` and `.max()`, `np.sum` and `.sum()`. They can disagree about missing values. December 1860 has no `tmin`. Here are three ways to average `tmin`, missing month included:

`np.mean` returns `nan`: one missing month spoils the whole result, as it did in `np.nan + 1`. `np.nanmean` skips missing values. pandas skips them by default, as yesterday's `.count()` showed. Neither choice is wrong, but you need to know which one you are getting.

Other NumPy functions have no pandas method: a column has `.mean()`, but no `.sqrt()`, `.exp()` or `.log()`. For those, call the NumPy function on the column. Like the arithmetic above, it works on every value at once, and gives back a column with the same dates.

For example, the most water vapour that air can hold rises steeply with its temperature. Meteorologists estimate it, in hectopascals (hPa), with the Magnus formula, which needs `np.exp`:

July 2026's air could hold about three times as much water as January's, which is why warmer air can bring heavier downpours.

### Fitting a straight line

NumPy fits lines too. `np.polyfit(x, y, 1)` finds the straight line that best fits the points (degree 1 means a straight line). It returns the slope and the intercept. Like most NumPy functions, it takes arrays or pandas columns.

Positive: summers have been warming. The x values are years, so the slope is in °C per year, and times ten gives °C per decade.

A single line through every summer since 1853 treats the whole record alike. Fit it again using only summers from 1970 onwards. A mask keeps those rows, as yesterday: `summer_tmax[summer_tmax["yyyy"] >= 1970]`.

About five times steeper. Draw both lines on one chart. A straight line's value in any year is `slope * year + intercept`, and, as with `(tmax + tmin) / 2`, one line of vectorised arithmetic works it out for every year at once. Keep hold of `fig` and `ax`: at the end of this section we add to the same chart.

A line through every summer since 1853 hides the fact that most of the warming is recent. The start year is a choice, and it changes the answer.

### How sure are we?

A few dozen noisy summers could slope upwards by chance. `scipy.stats.linregress` fits the same straight line as `np.polyfit`, and also says how far to trust it. SciPy, the scientific library built on NumPy, is one of the libraries you installed before the course.

Four numbers, in turn:

- **Slope:** 0.47 °C per decade, the same as `np.polyfit`.
- **Standard error** (`stderr`): how far off the slope could easily be. Roughly, the true rate is within two standard errors of it: here, somewhere between 0.25 and 0.69 °C per decade.
- **p-value:** if summers were not warming at all, how often would chance alone give a slope this steep? Less than once in 10,000, so chance alone is very unlikely to explain it.
- **r²:** the share of the ups and downs from one summer to the next that the line explains. Only a quarter: most of the difference between one summer and the next is weather, not the trend.

### Unusual for its time?

Summers are warming, so a record is less surprising than it used to be. A fairer question: how far is 2026 from what the recent trend would predict? This cell is filled in, because it has several new ideas at once. Read it a line at a time:

1. fit the line to 1970 to 2025, so 2026 cannot pull the line towards itself
2. use the line to predict 2026
3. measure each summer's gap from the line, called a **residual**, and take their standard deviation: the **residual standard deviation**
4. compare 2026's distance from the line with the residual standard deviation

In [ ]:
before = recent[recent["yyyy"] <= 2025]
trend_slope, trend_intercept = np.polyfit(before["yyyy"], before["tmax"], 1)
expected_2026 = trend_slope * 2026 + trend_intercept
residual_sd = (before["tmax"] - (trend_slope * before["yyyy"] + trend_intercept)).std()
tmax_2026 - expected_2026, residual_sd

The same comparison on the chart from above. This cell is filled in too, so read it a line at a time. The 1970 to 2025 line is carried on to 2026, dashed, over the years `np.arange(1970, 2027)`. `ax.fill_between(x, lower, upper)` shades the area between two lines: here a grey band one residual standard deviation either side of the trend, made see-through by `alpha=0.2`. The dotted red line, just `ax.plot` between two points, is the gap between 2026 and the trend. Last, `ax.set_xlim` zooms in on the years since 1960, and `bbox_to_anchor=(1, 1)` moves the legend, now too long to sit beside the data, outside the axes.

In [ ]:
years = np.arange(1970, 2027)
line = trend_slope * years + trend_intercept
ax.plot(years, line, color="black", linestyle="--", label="Trend fitted to 1970 to 2025")
ax.fill_between(years, line - residual_sd, line + residual_sd, color="grey", alpha=0.2,
                label="Residual standard deviation")
ax.scatter(2026, tmax_2026, color="red", zorder=3, label="2026 (provisional)")
ax.plot([2026, 2026], [expected_2026, tmax_2026], color="red", linestyle=":")
ax.set_xlim(1960, 2030)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
fig

2026 is about two and a half times as far from the trend as a typical summer. That ratio, the distance from what was expected divided by the residual standard deviation, is called a **z-score**. If the residuals follow the usual bell-shaped curve, fewer than 1 in 100 summers would land 2.5 or more above the line: unusual even allowing for the warming.

What a line can and cannot tell you:

- It **can** summarise the average rate of change over the years you chose.
- It **cannot** tell you why, or promise that the next ten years will follow it.
- 2026 is still **provisional**, and every 2026 temperature is estimated. The Met Office may revise them.

## Break

*11:25, 10 minutes*

## 4. A second station

*11:35, 35 minutes*

Is Oxford typical? Lerwick, in Shetland, is over 900 km further north (60.14°N, 1.18°W, 82 metres above sea level) and surrounded by sea. Its record starts in December 1930.

Yesterday's cleaning steps need to run again for a second file, so we turn them into a function, `load_station(path)`, that returns the cleaned DataFrame, and then move it into a file of its own. You do not need to type it from scratch: copy the cleaning cell from the top of this notebook, add a `def` line, indent everything, call the DataFrame `df` and add `return df`.

Does the function reproduce yesterday's result exactly?

Now the second station. Its file is in the `data` folder too:

In [ ]:
# Snapshot of https://www.metoffice.gov.uk/pub/data/weather/uk/climate/stationdata/lerwickdata.txt
LERWICK = DATA / "lerwickdata.txt"
LERWICK.exists()

Read the error from the bottom up: a sunshine value ending in `#` could not be read. The header of the raw file says what it means. At Lerwick, a `#` after a sunshine value means it came from an automatic sensor rather than the older recorder. Our function was written for Oxford, which never uses `#`.

A loud failure like this is a good thing: it stopped us before anything went wrong.

The fix follows yesterday's pattern: strip the `#` as well as the `*`, and keep the information in a `sun_auto` column, just as we kept the estimates. Copy your function into the next cell, add one line and change another:

### Moving the function into a file

`load_station` works now, but a function defined in a notebook only exists in that notebook, and only once its cell has run. Code you want to reuse belongs in a `.py` file next to the notebook, which any notebook or script in the folder can `import`, just as we import pandas.

`%%writefile helper.py`, on the first line of a cell, saves the rest of the cell to a file called `helper.py` instead of running it. A `.py` file can't see the notebook's variables, so it needs its own `import pandas as pd`, and the `COLUMNS` and `MEASURES` lists the function uses. Type the first two lines, copy `COLUMNS` and `MEASURES` from the top of the notebook, then paste your last `load_station` underneath.

Open `helper.py` from the file browser: an ordinary text file, with nothing notebook-specific in it. Now import the function from it. The import replaces the `load_station` defined in this notebook, so load both stations again with the imported one:

### Stacking the stations

To compare the stations, stack them into one table with a `station` column. `pd.concat` puts tables on top of each other, and `.assign` adds a column with the same value in every row. The result is **long** data: one row per station per month. Long data suits `groupby`.

`both` is worth keeping: the cleaned records of both stations in one table. `.to_csv` writes a DataFrame or Series to a CSV file, which opens in anything from Python to Excel, so a colleague can use the cleaned data without running any of this:

Now the summer mean max for both stations, on one `ax`. Was 2026 a record summer in Lerwick too?

**Bonus, if there's time:** the same summers two more ways, side by side in one figure: a box plot and a histogram.

No. 2026 is not even in Lerwick's top three summers. The heat of 2026 was a southern story, which is exactly why one station is not enough to talk about "the UK".

## 5. Combining the stations

*12:10, 30 minutes*

`both` is long: the stations stacked, one row per station per month. To read the stations side by side, **wide** data is easier: one column per station. There are two ways to get there, `unstack` and `merge`.

### From long to wide

`summers` has two levels of labels in its index, station and year. `unstack` moves the `station` level into the columns, as it moved the months into the columns for the heatmap in section 2. You will also meet `pivot`, which does the same job starting from ordinary columns, and `melt`, which goes the other way, from wide back to long.

Do the two stations have their hot summers in the same years? `.corr()` gives the **correlation** between each pair of columns: 1 means they rise and fall together exactly, 0 means no connection, and -1 means one rises as the other falls.

0.56: a hot summer in Oxford tends to be a warm one in Lerwick, but only loosely. A scatter plot shows what that looks like. Hover over the point furthest to the right:

2026 is far to the right, but only halfway up: the southern story again.

### Side by side by date: `merge`

`unstack` worked because both stations were already in one table. `merge` lines up two separate tables wherever their **key** columns match, here the year and the month. Columns with the same name in both tables get a suffix to tell them apart:

It starts in December 1930, not 1853. By default `merge` keeps only the months that are in both tables: an **inner** join. `how="left"` keeps every row of the left table, Oxford, and leaves Lerwick's columns missing before 1930. Count the rows each way:

Which rows survive is a decision, like `dropna` yesterday. With the stations side by side, one subtraction compares them month by month. How much warmer are Oxford's days than Lerwick's, at each time of year?

1.5 °C in December and January, but 8.1 °C in July. The sea round Shetland is slow to warm and slow to cool, so Lerwick's summers stay cool and its winters mild.

## 6. A first taste of machine learning

*12:40, 15 minutes*

So far we have fitted lines ourselves, with `np.polyfit` and `linregress`. **Machine learning** takes the same idea further: a model learns a pattern from data, and we judge it by how well it predicts data it has not seen. Python's usual library for it is scikit-learn, which the CfRR setup installed. Every model in it has the same three methods: `fit` (learn from data), `predict` (make predictions) and `score` (measure how good they are). Learn the pattern once and every model works the same way. The [CfRR Scikit-Learn page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_ScikitLearn.html) shows more models.

### The same line, a third way

A straight line is the simplest model there is. Fit block 3's 1970 to 2025 trend again, this time with scikit-learn. It expects the inputs as a table, with one row per example and one column per input. Double brackets pick columns as a table, so `before[["yyyy"]]` is a table with one column, the years:

The same slope as `np.polyfit`, in °C per year. Now `predict`: what does the line expect for 2026? The input is a table again, with the same column name and one row:

23.0 °C, the `expected_2026` from block 3. That is all a prediction is: what the pattern the model learned says should happen. The real 2026 came in 3.2 °C above it.

### Testing a model honestly

A model should be judged on data it has not seen. Hold some back as a **test set**, fit on the rest, and measure the error on the test set. With data that run through time, hold back the most recent: train on the past, test on the future. Here, fit on the summers before 2000 and test on 2000 onwards. `mean_absolute_error` is the average distance between the predictions and the real values, in °C:

A line trained on the summers up to 1999 misses the summers since by about 1.6 °C on average.

Why not hold back a random quarter of the summers instead? `train_test_split` does that:

A smaller error, and a bug that gives no error. Each test summer picked at random has training summers on both sides of it, so the model only fills in gaps rather than predicting the future, and the score flatters it. The honest number is the larger one, and it is the one to report. `random_state=42` makes the random split the same every time it runs, so the result can be reproduced.

### Oxford or Lerwick?

The line predicts a number. A **classifier** predicts a category instead: given one month's weather, which station is it? We use the months from 1931 onwards, when both stations recorded everything, and split by year again:

Before fitting anything, it is worth knowing what a score looks like with no model at all. The test set has as many Oxford months as Lerwick ones, so guessing at random would score 50%. A one-line rule, "if `tmax` is below 12 °C, it's Lerwick", does a little better:

These are **baselines**. A model is only worth having if it beats them. Now the model: a **decision tree**, which learns a series of yes/no questions about the inputs. For a classifier, `score` gives the share of test months it labels correctly:

Clearly better than both baselines, but far from perfect. Temperature alone mixes up the seasons: a Lerwick July and an Oxford April can look alike.

What else differs between a coastal climate and an inland one? The sea keeps Lerwick's nights mild and its days cool, so the gap between day and night temperatures is small. Add that gap as a new column and fit again:

A big jump. One well-chosen input, from knowing the subject, beat a more complicated model.

To see what the tree learned, draw it. `plot_tree` draws its questions on an `ax`, like any other chart. Each box shows the question it asks, how many training months reached it (`samples`), how many of those were Lerwick and how many Oxford (`value`), and the station it would guess (`class`):

Of everything we gave it, the tree asks about `trange` first: is the gap between day and night 5.45 °C or less? If so, the month is most likely Lerwick's.

## 7. Wrap: so, was summer 2026 unusual?

*12:55, 5 minutes*

This cell, filled in, pulls the numbers from both stations together:

In [ ]:
previous = summer_tmax[summer_tmax["yyyy"] < 2026]
record = previous.loc[previous["tmax"].idxmax()]
print(f"Summer 2026 mean max: {tmax_2026:.1f} °C")
print(f"Previous record: {record['tmax']:.1f} °C in {record['yyyy']:.0f}")
print(f"Above the 1970 to 2025 trend by {tmax_2026 - expected_2026:.1f} °C")
print(f"Residual standard deviation around the trend: {residual_sd:.1f} °C")
print(f"z-score: {(tmax_2026 - expected_2026) / residual_sd:.1f}")

lerwick_summers = summers["lerwick"]
warmer = (lerwick_summers > lerwick_summers[2026]).sum()
print(f"Lerwick summer 2026: {lerwick_summers[2026]:.1f} °C, with {warmer} of its {len(lerwick_summers)} summers warmer")

**Bonus, if there's time:** each station's summers as a histogram, with 2026 marked in red.

So what would we write, in a few sentences, and which caveats belong in it? One version:

> Summer 2026 was the hottest in Oxford's record, which goes back to 1853, by more than a degree, and well above what even the recent warming trend would predict. It was a southern extreme: in Lerwick, 900 km north, it was a warm summer but no record. The figures are still provisional, and the analysis should be rerun on a later snapshot once the Met Office confirms them.

### Reproducibility checklist

- **Pinned data:** a dated snapshot of the Met Office files, kept with the notebooks; [`data/README.md`](data/README.md) records where each file came from and its checksum.
- **Locked libraries:** the course repository's `uv.lock` records exact versions, and its `requirements.txt` is exported from it for pip users. For a project of your own, `pip freeze > requirements.txt` records what you have installed.
- **Fresh kernel:** the notebook runs top to bottom after a restart.
- **Seeds:** every random step has a `random_state`, so it gives the same answer every time.
- **Flags kept:** estimated, provisional and automatic-sensor values are marked, not dropped.
- **Checks:** count rows and missing values before and after each step, so a silent mistake shows up when the code meets new data.
- **Shared code in a file:** `load_station` lives in `helper.py`, imported wherever it is needed rather than copied between notebooks.

### Where next

- [Introduction to Machine Learning](https://coding-for-reproducible-research.github.io/CfRR_Courses/programme_information/introduction_to_machine_learning.html), for more models and how to test them properly
- [Introduction to Regression with R](https://coding-for-reproducible-research.github.io/CfRR_Courses/programme_information/intro_regression_analysis_with_r.html), for the statistics behind trend lines like ours: standard errors, p-values and r². It uses R, but the ideas are the same
- [Python Checking and Testing Code](https://coding-for-reproducible-research.github.io/CfRR_Courses/short_courses/python_testing.html), for turning checks like ours into tests that run every time
- [Version Control](https://coding-for-reproducible-research.github.io/CfRR_Courses/programme_information/version_control.html)